In [ ]:
import torch
import numpy as np
import pandas as pd
import config
from torch.utils.data import DataLoader
from preprocess import year_mask
from vae_train import VAEConv3D, vae_loss, run_sweep, SampleDS, train_vae_final

DATA = "/file/pathway"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
if device.type == "cuda":
    print("gpu:", torch.cuda.get_device_name(0))

# Data Splitting

In [ ]:
X = np.load(f"{DATA}/samples.npy", mmap_mode="r")
dates = np.load(f"{DATA}/sample_dates.npy", allow_pickle=True)

tr = SampleDS(X, year_mask(dates, *config.TRAIN_YEARS)) 
va = SampleDS(X, year_mask(dates, *config.VAL_YEARS))
te = SampleDS(X, year_mask(dates, *config.TEST_YEARS))
print(f"train {len(tr)} | val {len(va)} | test {len(te)}")

train_loader = DataLoader(tr, batch_size=32, shuffle=True, drop_last=True, num_workers=2)
val_loader = DataLoader(va, batch_size=32, num_workers=2)
test_loader = DataLoader(te, batch_size=32, num_workers=2)

# Instatiating the Model

In [ ]:
model = VAEConv3D(
    input_dim=7,
    hidden_dim=256,
    latent_dim=128,
    input_shape=(11, config.TARGET_H, config.TARGET_W),
    kernel_size=(3, 3, 3),
    stride=(1, 1, 1),
    padding=(1, 1, 1),
    apply_sigmoid=False,
).to(device)

print("flatten_dim:", model.flatten_dim, "| bottleneck (t,h,w):", (model.tl, model.hl, model.wl))

In [ ]:
# MODEL SMOKE TEST
x = torch.randn(4, 7, 11, config.TARGET_H, config.TARGET_W, device=device)

model.train()
recon, mu, logvar, z = model(x)
print("input:  ", tuple(x.shape))
print("recon:  ", tuple(recon.shape))
print("mu:     ", tuple(mu.shape))
print("logvar: ", tuple(logvar.shape))
print("z:      ", tuple(z.shape))

loss, recon_l, kld = vae_loss(recon, x, mu, logvar, beta=0.5)
loss.backward()
print("loss:", loss.item(), "| recon:", recon_l.item(), "| kld:", kld.item())

# Model Training

In [ ]:
# First Stage (Latent Dimension and Beta)

results = run_sweep(train_loader, val_loader, device,
    latents=[64, 128, 256], betas=[0.1, 0.25, 0.5], epochs=50)

df = pd.DataFrame(results).sort_values(["active_dims", "val_recon_per_elem"],
    ascending=[False, True])

df

In [ ]:
# Second Stage (Hidden Layers and Learning Rate)

stage2 = []
for hidden in [128, 256]:
    for lr in [1e-3, 1e-4]:
        stage2 += run_sweep(train_loader, val_loader, device,
                            latents=[128], betas=[0.1],
                            epochs=50, lr=lr, hidden=hidden)

df2 = pd.DataFrame(stage2).sort_values(["active_dims", "val_recon_per_elem"],
                                       ascending=[False, True])

df2

In [ ]:
# Final Re-training

torch.manual_seed(42)
np.random.seed(42)

final_model = VAEConv3D(
    input_dim=7, hidden_dim=256, latent_dim=128,
    input_shape=(11, config.TARGET_H, config.TARGET_W),
    kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1),
    apply_sigmoid=False,
).to(device)

final_model = train_vae_final(final_model, train_loader, val_loader, device,
    epochs=100, lr=1e-3, clip=1.0, beta_max=0.1)

torch.save(final_model.state_dict(), f"{DATA}/vae_conus_L64_b01.pth")